In [2]:
import torch

import torch.nn as nn

import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np

from tqdm import tqdm
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [3]:
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

In [ ]:
class DummyDataset(Dataset):
    def __init__(self):
        X = ...
        y = ...
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, index):
        return self.X[index], self.y[index]
    
dataset = DummyDataset()

train_size = ...
test_size = ...

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

In [ ]:
batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
class CNN(nn.Module):
    def __init__(self, input_shape=(3,28,28), num_classes=10):
        super.__init__()

        in_channels = input_shape[0]

        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16,3),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16,32,3),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        with torch.no_grad():
            dummy = torch.zeros(1, *input_shape)
            dummy_out = self.features(dummy)
            n_features = dummy_out.numel()

        self.classifier = nn.Sequential(
            nn.Linear(n_features, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

In [ ]:
model = CNN().to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()

    running_loss = correct = total = 0

    for X, y in loader: 
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X.size(0)

        _, predicted = torch.max(outputs, 1)

        correct =